# 本地IM/IH分钟行情适用性检查

187文件、53组同月合约；无买卖盘报价。原同步相对高点模型缺输入。先统一分钟标签、隔离缺口，再决定独立的收盘价研究规则。

本笔记默认重放2026-10-16已完成的原始扫描和独立复核快照，不重新读取源行情；不是回测。公开笔记不包含原始价格、账户资料或绝对本地路径。

## 输入、环境与重跑方式

从仓库根目录或research目录运行，Python和pandas。原始数据不随仓库分发；可用环境变量指定数据根目录。RUN_FULL_SCAN=True重扫源文件但只在内存保留新结果，不覆盖历史快照；REVERIFY_SOURCES=True只重读并核验来源。默认输出明确是快照重放。

In [1]:
from pathlib import Path
import json, platform, sys
import pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / "data").is_dir() and ROOT.name == "research":
    ROOT = ROOT.parent
assert (ROOT / "research/local_market_data_quality.py").is_file(), "请从仓库或research目录运行"
sys.path.insert(0, str(ROOT / "research"))
from local_market_data_quality import audit
from verify_local_market_data import verify
RUN_FULL_SCAN = False
REVERIFY_SOURCES = False
inventory = json.loads((ROOT / "data/local_market_data_inventory_20261016.json").read_text(encoding="utf8"))
verification = json.loads((ROOT / "data/local_market_data_verification_20261016.json").read_text(encoding="utf8"))
if RUN_FULL_SCAN:
    inventory = audit()
if RUN_FULL_SCAN or REVERIFY_SOURCES:
    verification = verify(inventory)
print({"mode": "source recheck" if RUN_FULL_SCAN or REVERIFY_SOURCES else "recorded snapshot replay", "python": platform.python_version(), "pandas": pd.__version__, "inventory_checked_at": inventory["checked_at"], "verification_checked_at": verification["checked_at"]})

{'mode': 'recorded snapshot replay', 'python': '3.12.14', 'pandas': '3.0.1', 'inventory_checked_at': '2026-10-16T00:46:39.010341+00:00', 'verification_checked_at': '2026-10-16T00:51:21.870104+00:00'}


## 文件粒度与输入适用性

一行是来源文件的一根分钟记录，不是同步报价。valid_unique_time_rows仅表示时间可解析、价格有效且时间唯一，未包含完整交易日历或报价时效认证。

In [2]:
print(json.dumps(inventory["summary"], ensure_ascii=False, indent=2))
print("原同步报价模型输入可用:", inventory["summary"]["files_with_quote_columns"] > 0)
assert not verification["hash_mismatches"], "来源发生变化，应重建审计版本"
print("独立复核指纹数:", verification["hashes_rechecked"])

{
  "primary_raw_files": 180,
  "im_raw_files": 53,
  "ih_raw_files": 127,
  "same_month_pairs": 53,
  "files_profiled": 187,
  "rows_profiled": 3625310,
  "files_with_quote_columns": 0,
  "source_files_unchanged": true,
  "raw_rows_with_invalid_prices": 48,
  "raw_duplicate_time_extra_rows": 0
}
原同步报价模型输入可用: False
独立复核指纹数: 187


## 标签、缺失与异常

网格仅用于诊断，不能据此直接平移或补齐。交并比只统计共同有记录日期；另列整日单边缺口，避免掩盖。

In [3]:
p = inventory["continuous_pair"]
print({k:p[k] for k in ("shared_observed_days", "common_labels", "union_labels", "common_over_union", "im_only_labels_on_shared_days", "ih_only_labels_on_shared_days", "month_mismatch_rows")})
print("逐日网格分布:", {k:d["day_grid_counts"] for k,d in verification["continuous_day_grids"].items()})
print("IM有而IH拼接无的日期:", verification["im_observed_dates_absent_from_ih"])
print("独立CSV异常:", verification["independent_csv_anomaly_counts"])
print("KQ.m时钟诊断:", inventory["main_clock_diagnostics"])

{'shared_observed_days': 1022, 'common_labels': 245186, 'union_labels': 245298, 'common_over_union': 0.9995434125023441, 'im_only_labels_on_shared_days': 94, 'ih_only_labels_on_shared_days': 18, 'month_mismatch_rows': 0}
逐日网格分布: {'IM': {'start': 1001, 'end': 23}, 'IH': {'start': 2489, 'other': 3, 'end': 30}}
IM有而IH拼接无的日期: ['2026-08-06', '2026-09-10']
独立CSV异常: {'actual_zero_or_negative_ohlc_rows': 0, 'nonfinite_ohlc_rows': 48, 'all_fields_empty_rows': 0, 'empty_datetime_rows': 0, 'structurally_short_rows': 47, 'datetime_contains_nul_rows': 47}
KQ.m时钟诊断: [{'symbol': 'IM', 'as_stored_label_intersection': 0, 'plus_8_hours_label_intersection': 10000, 'interpretation': 'clock-alignment diagnostic only; not a timezone proof or automatic conversion'}, {'symbol': 'IH', 'as_stored_label_intersection': 0, 'plus_8_hours_label_intersection': 10000, 'interpretation': 'clock-alignment diagnostic only; not a timezone proof or automatic conversion'}]


## 三个案例的覆盖核对

只核行数、价格检查与同标签覆盖；不生成方向、触发、盈亏或可成交结论。2024年2月5日的事前DetectionTime仍缺。

In [4]:
for case in verification["independent_case_checks"]:
    assert case["agrees_with_inventory"]
    print(case)
print("来源均为本地缓存，供数时区、bar含义、真实报价及官方日历尚未逐段认证。")

{'date': '2024-02-05', 'month': '2402', 'im_rows': 240, 'ih_rows': 240, 'shared_labels': 240, 'duplicates': 0, 'agrees_with_inventory': True}
{'date': '2025-02-05', 'month': '2502', 'im_rows': 240, 'ih_rows': 240, 'shared_labels': 240, 'duplicates': 0, 'agrees_with_inventory': True}
{'date': '2025-04-07', 'month': '2504', 'im_rows': 240, 'ih_rows': 240, 'shared_labels': 240, 'duplicates': 0, 'agrees_with_inventory': True}
来源均为本地缓存，供数时区、bar含义、真实报价及官方日历尚未逐段认证。


## 结论与下一步

现有分钟行情可供后续独立收盘价规则研究，但先核供数/时区/标签并隔离异常与缺口。原QH同步报价模型仍缺逐笔买卖盘输入，分钟相对High/Low不能从独立两腿极值拼接。

检查脚本只读源数据，不调用生产事件库、联网、回测或执行。所有笔记代码已由Python顺序运行；本环境没有Jupyter/nbformat，未做Jupyter内核及官方schema验证。

## 分钟数据隔离规则 V0.1

本轮54文件、519890条记录中48条逻辑隔离；其余记录只通过结构检查，未确认供数时区或bar含义。原文件没有修改。下面重放本轮结果，不复扫此前全部187份。

In [5]:
from minute_data_admission import build_dossier
admission_contract = json.loads((ROOT / "data/minute_data_contract_v0_1.json").read_text(encoding="utf8"))
admission = json.loads((ROOT / "data/minute_data_admission_20261016.json").read_text(encoding="utf8"))
RECHECK_ADMISSION_SOURCES = False
if RECHECK_ADMISSION_SOURCES:
    import os
    source_root = Path(os.environ.get("IM_IH_AUDIT_SOURCE_ROOT", ROOT.parent / "数据/期货/股指期货"))
    admission = build_dossier(inventory, admission_contract, source_root)
print("本单元模式:", "重新核源" if RECHECK_ADMISSION_SOURCES else "已保存诊断快照重放")
print(admission["summary"])
assert len(admission["quarantine_index"]) == admission["summary"]["quarantined_rows"]
print("隔离原因可以重叠，不能将原因数相加当记录数。")

本单元模式: 已保存诊断快照重放
{'files_rechecked': 54, 'rows_read': 519890, 'quarantined_rows': 48, 'structurally_valid_rows': 519842, 'case_days': 3, 'gap_days': 2, 'research_admitted_case_days': 0, 'entry_allowed': False}
隔离原因可以重叠，不能将原因数相加当记录数。


In [6]:
for item in admission["case_days"] + admission["gap_days"]:
    print({k:item[k] for k in ("date", "im_contract", "ih_contract", "common_structurally_valid_labels", "im_only_labels", "ih_only_labels", "research_admitted")})
print("隔离索引示例（仅前2条，无原始价格）:", admission["quarantine_index"][:2])

{'date': '2024-02-05', 'im_contract': 'IM2402', 'ih_contract': 'IH2402', 'common_structurally_valid_labels': 240, 'im_only_labels': 0, 'ih_only_labels': 0, 'research_admitted': False}
{'date': '2025-02-05', 'im_contract': 'IM2502', 'ih_contract': 'IH2502', 'common_structurally_valid_labels': 240, 'im_only_labels': 0, 'ih_only_labels': 0, 'research_admitted': False}
{'date': '2025-04-07', 'im_contract': 'IM2504', 'ih_contract': 'IH2504', 'common_structurally_valid_labels': 240, 'im_only_labels': 0, 'ih_only_labels': 0, 'research_admitted': False}
{'date': '2026-08-06', 'im_contract': 'IM2608', 'ih_contract': 'IH2608', 'common_structurally_valid_labels': 0, 'im_only_labels': 240, 'ih_only_labels': 0, 'research_admitted': False}
{'date': '2026-09-10', 'im_contract': 'IM2609', 'ih_contract': 'IH2609', 'common_structurally_valid_labels': 0, 'im_only_labels': 240, 'ih_only_labels': 0, 'research_admitted': False}
隔离索引示例（仅前2条，无原始价格）: [{'file_id': 'primary/raw/IM2208_1分钟.csv', 'csv_record_ordin

### 准入边界

三个完整案例日仍为DIAGNOSTIC_ONLY；来源、时区、bar定义及官方日历未知。2026-08-06和09-10对应的IH原始缓存也无当日记录，不以拼接或旧副本补齐。不可解析时间的47条记录仅在文件级定位。

下一步是对案例时间口径补依据，并将分钟收盘价观察规则单独编号。本笔记不包含信号或执行代码。全量196项应用/离线检查测试通过。